In [1]:
from pathlib import Path
import napari
from skimage import io
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import skimage
from sklearn.metrics import confusion_matrix
import pickle
import h5py
from tqdm.notebook import tqdm
import networkx as nx


from cell_paint_seg.utils import (
    get_id_to_path,
    get_id_from_name_96,
    combine_soma_cell_labels,
    combine_soma_nucleus_labels,
    check_valid_labels,
)

In [3]:
data_dir = Path("/Users/thomasathey/Documents/shavit-lab/fraenkel/data/aneesh/2025_04_nuc-stains/raw")
cp_order = {1:"ER",2:"DNA",3:"Actin",4:"RNA",5:"Golgi"}
antibody_order = {1: "DAPI", 3: "NeuN"}

In [14]:
# list all tifs in the data directory
reg_path = Path("/Users/thomasathey/Documents/shavit-lab/fraenkel/data/aneesh/2025_04_nuc-stains/tifs_for_reg")

tif_files = list(data_dir.glob("*.tif")) + list(reg_path.glob("*.tif"))
                
im_paths = {}

for tif in tif_files:
    name = tif.stem 

    if "reg" in name:
        id = "s" + name.split("_")[0]
        channel_name = "NeuN_reg"
    elif "antibody" in name:
        id = name.split("_")[-1].split("c")[0]
        c = int(name[-1])
        channel_name = antibody_order[c]
    else:
        id = name.split("_")[-1].split("c")[0]
        c = int(name[-1])
        channel_name = cp_order[c]
    

    if id not in im_paths.keys():
        im_paths[id] = {channel_name: tif}
    else:
        cur_dict = im_paths[id]
        cur_dict[channel_name] = tif
        im_paths[id] = cur_dict
im_paths

{'s32': {'Golgi': PosixPath('/Users/thomasathey/Documents/shavit-lab/fraenkel/data/aneesh/2025_04_nuc-stains/tifs_for_reg/Cell_Paint__s32c5.tif'),
  'RNA': PosixPath('/Users/thomasathey/Documents/shavit-lab/fraenkel/data/aneesh/2025_04_nuc-stains/tifs_for_reg/Cell_Paint__s32c4.tif'),
  'Actin': PosixPath('/Users/thomasathey/Documents/shavit-lab/fraenkel/data/aneesh/2025_04_nuc-stains/tifs_for_reg/Cell_Paint__s32c3.tif'),
  'DNA': PosixPath('/Users/thomasathey/Documents/shavit-lab/fraenkel/data/aneesh/2025_04_nuc-stains/tifs_for_reg/Cell_Paint__s32c2.tif'),
  'ER': PosixPath('/Users/thomasathey/Documents/shavit-lab/fraenkel/data/aneesh/2025_04_nuc-stains/tifs_for_reg/Cell_Paint__s32c1.tif'),
  'NeuN': PosixPath('/Users/thomasathey/Documents/shavit-lab/fraenkel/data/aneesh/2025_04_nuc-stains/tifs_for_reg/antibody__s32c3.tif'),
  'DAPI': PosixPath('/Users/thomasathey/Documents/shavit-lab/fraenkel/data/aneesh/2025_04_nuc-stains/tifs_for_reg/antibody__s32c1.tif'),
  'NeuN_reg': PosixPath('/

In [ ]:
out_dir = Path("/Users/thomasathey/Documents/shavit-lab/fraenkel/data/aneesh/2025_04_nuc-stains/tifs_for_annot")

for id, paths in im_paths.items():
    ims = []

    for key in ["NeuN_reg", "DNA", "RNA"]:
        im = io.imread(paths[key])
        im = np.amax(im, axis=-1)
        ims.append(im)

    ims = np.stack(ims, axis=-1)

    # save the image
    out_path = out_dir / f"{id}.tif"
    io.imsave(out_path, ims)
    

In [19]:
seg_path = Path("/Users/thomasathey/Documents/shavit-lab/fraenkel/data/aneesh/2025_04_nuc-stains/tifs_for_annot")

# get all files that have the word "Cell"
seg_files = list(seg_path.glob("*Cell*.tif"))

for seg_file in seg_files:
    id = seg_file.stem.split("_")[-1][:3]
    newname = seg_file.parent / f"{id}_nuc.tif"

    # rename the file
    seg_file.rename(newname)
    print(f"Renamed {seg_file} to {newname}")


Renamed /Users/thomasathey/Documents/shavit-lab/fraenkel/data/aneesh/2025_04_nuc-stains/tifs_for_annot/Cell_Paint__s27c9.tif to /Users/thomasathey/Documents/shavit-lab/fraenkel/data/aneesh/2025_04_nuc-stains/tifs_for_annot/s27_nuc.tif
Renamed /Users/thomasathey/Documents/shavit-lab/fraenkel/data/aneesh/2025_04_nuc-stains/tifs_for_annot/Cell_Paint__s25c9.tif to /Users/thomasathey/Documents/shavit-lab/fraenkel/data/aneesh/2025_04_nuc-stains/tifs_for_annot/s25_nuc.tif
Renamed /Users/thomasathey/Documents/shavit-lab/fraenkel/data/aneesh/2025_04_nuc-stains/tifs_for_annot/Cell_Paint__s21c9.tif to /Users/thomasathey/Documents/shavit-lab/fraenkel/data/aneesh/2025_04_nuc-stains/tifs_for_annot/s21_nuc.tif
Renamed /Users/thomasathey/Documents/shavit-lab/fraenkel/data/aneesh/2025_04_nuc-stains/tifs_for_annot/Cell_Paint__s23c9.tif to /Users/thomasathey/Documents/shavit-lab/fraenkel/data/aneesh/2025_04_nuc-stains/tifs_for_annot/s23_nuc.tif
Renamed /Users/thomasathey/Documents/shavit-lab/fraenkel/dat